In [1]:
import pandas as pd

df = pd.read_csv(
    "soc-redditHyperlinks-body.tsv",
    sep="\t")

df.head()

,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,leagueoflegends,teamredditteams,1u4nrps,2013-12-31 16:39:58,1,"345.0,298.0,0.75652173913,0.0173913043478,0.08..."
1,theredlion,soccer,1u4qkd,2013-12-31 18:18:37,-1,"101.0,98.0,0.742574257426,0.019801980198,0.049..."
2,inlandempire,bikela,1u4qlzs,2014-01-01 14:54:35,1,"85.0,85.0,0.752941176471,0.0235294117647,0.082..."
3,nfl,cfb,1u4sjvs,2013-12-31 17:37:55,1,"1124.0,949.0,0.772241992883,0.0017793594306,0...."
4,playmygame,gamedev,1u4w5ss,2014-01-01 02:51:13,1,"715.0,622.0,0.777622377622,0.00699300699301,0...."


In [2]:
print(df.shape)

(286561, 6)


In [3]:
df = df[[
    "SOURCE_SUBREDDIT",
    "TARGET_SUBREDDIT",
    "POST_ID",
    "TIMESTAMP",
    "LINK_SENTIMENT"
]]

df.head()

,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT
0,leagueoflegends,teamredditteams,1u4nrps,2013-12-31 16:39:58,1
1,theredlion,soccer,1u4qkd,2013-12-31 18:18:37,-1
2,inlandempire,bikela,1u4qlzs,2014-01-01 14:54:35,1
3,nfl,cfb,1u4sjvs,2013-12-31 17:37:55,1
4,playmygame,gamedev,1u4w5ss,2014-01-01 02:51:13,1


In [5]:
edges = (
    df.groupby(
        ["SOURCE_SUBREDDIT",
         "TARGET_SUBREDDIT",
         "LINK_SENTIMENT"]
    )
    .size()
    .reset_index(name="Weight")
)

In [6]:
import networkx as nx

G = nx.from_pandas_edgelist(
    edges,
    source="SOURCE_SUBREDDIT",
    target="TARGET_SUBREDDIT",
    create_using=nx.DiGraph()
)

In [7]:
degree = dict(G.degree())

In [8]:
degree_df = pd.DataFrame(
    degree.items(),
    columns=["Subreddit","Degree"]
)

In [9]:
top_nodes = degree_df.nlargest(
    1000,
    "Degree"
)["Subreddit"]

In [10]:
filtered_edges = edges[
    edges["SOURCE_SUBREDDIT"].isin(top_nodes) &
    edges["TARGET_SUBREDDIT"].isin(top_nodes)
]

In [11]:
nodes = pd.DataFrame({
    "Id": pd.unique(
        filtered_edges[
            ["SOURCE_SUBREDDIT",
             "TARGET_SUBREDDIT"]
        ].values.ravel()
    )
})

In [12]:
nodes["Label"] = nodes["Id"]

In [13]:
filtered_edges = filtered_edges.rename(columns={
    "SOURCE_SUBREDDIT":"Source",
    "TARGET_SUBREDDIT":"Target",
    "LINK_SENTIMENT":"Sentiment"
})

In [14]:
nodes.to_csv(
    "reddit_nodes.csv",
    index=False
)

In [15]:
filtered_edges.to_csv(
    "reddit_edges.csv",
    index=False
)

In [16]:
import networkx as nx

G = nx.from_pandas_edgelist(
    edges,
    source="SOURCE_SUBREDDIT",
    target="TARGET_SUBREDDIT",
    create_using=nx.DiGraph()
)

In [17]:
degree = dict(G.degree())

degree_df = (
    pd.DataFrame(degree.items(), columns=["Subreddit", "Degree"])
    .sort_values("Degree", ascending=False)
)

In [18]:
top_nodes = set(degree_df.head(5000)["Subreddit"])

In [19]:
filtered_edges = edges[
    edges["SOURCE_SUBREDDIT"].isin(top_nodes) &
    edges["TARGET_SUBREDDIT"].isin(top_nodes)
]

In [20]:
nodes = pd.DataFrame({
    "Id": sorted(top_nodes)
})

nodes["Label"] = nodes["Id"]

In [21]:
print("Nodes:", len(nodes))
print("Edges:", len(filtered_edges))

Nodes: 5000
Edges: 87583


In [22]:
import pandas as pd

# Extract all unique nodes from the filtered edge list
nodes = pd.DataFrame({
    "Id": pd.unique(
        filtered_edges[
            ["SOURCE_SUBREDDIT", "TARGET_SUBREDDIT"]
        ].values.ravel()
    )
})

nodes["Label"] = nodes["Id"]

print("Number of nodes:", len(nodes))
nodes.head()

Number of nodes: 4997


,Id,Label
0,100daysofketo,100daysofketo
1,bodyweightfitness,bodyweightfitness
2,documentaries,documentaries
3,keto,keto
4,loseit,loseit


In [23]:
edges_gephi = filtered_edges.rename(columns={
    "SOURCE_SUBREDDIT": "Source",
    "TARGET_SUBREDDIT": "Target",
    "LINK_SENTIMENT": "Sentiment"
})

edges_gephi.head()

,Source,Target,Sentiment,Weight
6,100daysofketo,bodyweightfitness,1,1
7,100daysofketo,documentaries,1,1
8,100daysofketo,keto,1,5
9,100daysofketo,loseit,1,1
10,100daysofketo,xxketo,1,2


In [24]:
nodes.to_csv("reddit_nodes_5000.csv", index=False)
edges_gephi.to_csv("reddit_edges_5000.csv", index=False)